# TP 2 : agrandir la base, puis la réduire

Le même modèle linéaire, mais nourri d'autres colonnes : d'abord beaucoup plus, ensuite beaucoup
moins.

## 2.0 Projeter les entrées dans un espace plus grand

Une droite ne peut pas suivre une tendance courbe. On n'abandonne pas le modèle linéaire pour
autant : on remplace $x$ par une **base de fonctions** $\phi(x)$, fixée avant l'ajustement, et on
ajuste linéairement dans cet espace plus grand :

$$h_w(x) = w^T \phi(x) = \sum_{j=0}^{M-1} w_j \phi_j(x), \qquad \phi_0(x) = 1 .$$

Avec $\phi(x) = (1, x)$ on retrouve exactement le modèle de tout à l'heure. Avec tous les monômes
jusqu'au degré $d$ ($x_1$, $x_1 x_2$, $x_3^2$, …), on obtient la **régression polynomiale**.

On empile les $\phi(x^{(i)})^T$ en lignes de $\Phi \in \mathbb{R}^{m \times M}$, et **rien ne
change** dans l'ajustement : $w = (\Phi^T \Phi)^{-1}\Phi^T y$, $w = (\Phi^T \Phi + \lambda
I)^{-1}\Phi^T y$.

Les colonnes de $\Phi$ n'ont plus la même échelle, un produit de trois mesures est bien plus
petit qu'une mesure seule, donc on les centre et on les réduit avant d'ajuster. Sinon la
pénalité $\lambda$ frapperait des colonnes incomparables.

**Q1.** La base de degré 2 compte 66 colonnes et celle de degré 3 en compte 286, pour 353 patients
d'entraînement. Avec 286 coefficients pour 353 points, le modèle peut coller de très près à chacun de
ses points d'entraînement. Qu'en concluez-vous sur ce que mesure son erreur d'entraînement ?

**Votre réponse :**



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

data = load_diabetes(as_frame=True)
feature_names = list(data.data.columns)
X, y = data.data.values, data.target.values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)


def design_matrix(Phi):
    """Add the dummy column phi_0 = 1 in front of a basis."""
    return np.hstack([np.ones((len(Phi), 1)), Phi])


def predict(Phi, w):
    return design_matrix(Phi) @ w


def polynomial_basis(degree, X_reference):
    """Monomials up to `degree`, centred and scaled on X_reference. Returns the function phi."""
    expand = PolynomialFeatures(degree, include_bias=False).fit(X_reference)
    scale = StandardScaler().fit(expand.transform(X_reference))
    # TODO: return a function that applies `expand` then `scale` to any array of inputs
    ...


phi_2 = polynomial_basis(2, X_train)
assert phi_2(X_train).shape == (353, 65)
assert np.allclose(phi_2(X_train).mean(axis=0), 0.0)
assert np.allclose(phi_2(X_train).std(axis=0), 1.0)
assert not np.allclose(phi_2(X_test).mean(axis=0), 0.0)

for degree in (1, 2, 3):
    phi = polynomial_basis(degree, X_train)
    print(f"degré {degree} : M = {phi(X_train).shape[1] + 1} colonnes pour "
          f"{len(X_train)} patients")

## 2.1 Ce que fait une base trop grande

**Q2.** La base de degré 2 contient celle de degré 1. Quand on passe du degré 1 au degré 2, le
$R^2$ d'entraînement peut-il descendre, ou ne peut-il que monter ? Qu'en concluez-vous sur la
possibilité de choisir $M$ avec ce chiffre ?

**Votre réponse :**



In [ ]:
def least_squares(Phi, y):
    """w = (Phi^T Phi)^-1 Phi^T y, via lstsq: at degree 3 the matrix is close to singular."""
    # TODO: return the least-squares weights, with np.linalg.lstsq(..., rcond=None)[0]
    ...


def r2(y, y_hat):
    # TODO: 1 - sum of squared residuals / sum of squared deviations from the mean of y
    ...


def rmse(y, y_hat):
    # TODO: root mean squared error
    ...


rows = []
for degree in (1, 2, 3):
    phi = polynomial_basis(degree, X_train)
    Phi_train, Phi_test = phi(X_train), phi(X_test)
    w = least_squares(Phi_train, y_train)
    rows.append({
        "M": Phi_train.shape[1] + 1,
        "cond(ΦᵀΦ)": np.linalg.cond(design_matrix(Phi_train).T @ design_matrix(Phi_train)),
        "R² entraînement": r2(y_train, predict(Phi_train, w)),
        "R² test": r2(y_test, predict(Phi_test, w)),
        "RMSE test": rmse(y_test, predict(Phi_test, w)),
    })

table = pd.DataFrame(rows, index=["degré 1", "degré 2", "degré 3"])
assert np.isclose(table.loc["degré 1", "R² test"],
                  LinearRegression().fit(X_train, y_train).score(X_test, y_test))
print(table.to_string(formatters={"cond(ΦᵀΦ)": "{:.2e}".format, "R² entraînement": "{:.3f}".format,
                                  "R² test": "{:.3f}".format, "RMSE test": "{:.1f}".format}))
print(f"\nRMSE de test du modèle qui prédit toujours la moyenne d'entraînement : "
      f"{rmse(y_test, np.full(len(y_test), y_train.mean())):.1f}")

**Q3.** Dans le tableau, le degré 3 affiche le meilleur $R^2$ d'entraînement, 0,887, et un RMSE de
test de 184,3 ; la ligne en dessous donne 71,7 pour le modèle qui répond toujours la moyenne. Le
degré 3 prédit-il mieux ou moins bien que ce modèle constant ? Qu'en concluez-vous sur son 0,887 ?

**Votre réponse :**



## 2.2 Le $+\lambda I$ rattrape la base

**Q4.** Le degré 3 a 286 colonnes pour 353 patients, et la droite obtient 0,332 de $R^2$ de test
dans le tableau ci-dessus. En ajoutant $\lambda I$, jusqu'où le $R^2$ de test du degré 3 peut-il
remonter : en dessous de 0,332, à peu près à 0,332, ou nettement au-dessus ?

**Votre réponse :**



In [ ]:
def ridge_equation(Phi, y, lam):
    """w = (Phi^T Phi + lam I)^-1 Phi^T y. The bias is not penalised."""
    A = design_matrix(Phi)
    penalty = lam * np.eye(A.shape[1])
    penalty[0, 0] = 0.0
    # TODO: solve (A^T A + penalty) w = A^T y
    ...


reference = Ridge(alpha=1.0).fit(phi_2(X_train), y_train)
assert np.isclose(ridge_equation(phi_2(X_train), y_train, 1.0)[0], reference.intercept_)
assert np.allclose(ridge_equation(phi_2(X_train), y_train, 1.0)[1:], reference.coef_)

lambdas = np.logspace(-3, 4, 15)
phi3 = polynomial_basis(3, X_train)
Phi3_train, Phi3_test = phi3(X_train), phi3(X_test)

r2_train_path, r2_test_path = [], []
for lam in lambdas:
    # TODO: ridge weights on (Phi3_train, y_train) with this lam, then append its R2
    # on the training set and on the test set
    ...

assert len(r2_test_path) == 15
assert np.isclose(r2_test_path[0], r2(y_test, predict(Phi3_test, ridge_equation(Phi3_train, y_train, lambdas[0]))))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(lambdas, r2_train_path, "o-", ms=3, label="R² d'entraînement, degré 3")
ax.plot(lambdas, r2_test_path, "o-", ms=3, label="R² de test, degré 3")
ax.axhline(r2(y_test, np.full(len(y_test), y_train.mean())),
           color="gray", ls=":", lw=1, label="prédire la moyenne")
ax.axhline(0.332, color="crimson", ls="--", lw=1, label="la droite (degré 1, λ = 0)")
ax.set_xscale("log")
ax.set_ylim(-1, 0.7)
ax.set_xlabel("λ")
ax.set_ylabel("R²")
ax.legend(fontsize=8)
plt.show()

best = int(np.argmax(r2_test_path))
print(f"meilleur R² de test du degré 3 : {r2_test_path[best]:.3f} à λ = {lambdas[best]:.3g}")

**Q5.** Le meilleur $R^2$ de test du degré 3 pénalisé est 0,345, et le trait rouge de la figure,
la droite, est à 0,332. Le degré 3 a-t-il trouvé une courbure que la droite ignore, ou $\lambda$ lui
a-t-il seulement retiré la capacité de passer par ses points d'entraînement ?

**Votre réponse :**



## 2.3 La base et la pénalité se choisissent ensemble

Ce qui s'**apprend** des données, ce sont les paramètres $w$. Ce qui se **choisit** avant, ce sont
la famille de base $\phi$, sa taille $M$, et la pénalité $\lambda$ : trois hyperparamètres. On les
choisit ensemble, sur une grille, en gardant de côté une partie de l'entraînement : la
**validation**. Le test reste fermé.

**Q6.** Trois degrés et quinze $\lambda$ : quarante-cinq modèles, tous jugés sur la même validation.
Va-t-elle retenir le degré 3 fortement pénalisé, qui atteignait 0,345 de $R^2$ de test en 2.2, ou le
degré 1 ? Justifiez en une phrase.

**Votre réponse :**



In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=0)
degrees = [1, 2, 3]
validation_rmse = np.zeros((len(degrees), len(lambdas)))

for i, degree in enumerate(degrees):
    # the basis is fitted on the training part only: fitting it on the validation part too
    # would let the validation set influence the model
    phi = polynomial_basis(degree, X_fit)
    Phi_fit, Phi_val = phi(X_fit), phi(X_val)
    for j, lam in enumerate(lambdas):
        # TODO: ridge weights on (Phi_fit, y_fit), then store their RMSE on (Phi_val, y_val)
        # in validation_rmse[i, j]
        ...

assert validation_rmse.shape == (3, 15)
i_best, j_best = np.unravel_index(np.argmin(validation_rmse), validation_rmse.shape)
degree_best, lambda_best = degrees[i_best], lambdas[j_best]

print(pd.DataFrame(validation_rmse.round(1), index=[f"degré {d}" for d in degrees],
                   columns=[f"{lam:.3g}" for lam in lambdas]).to_string())
print(f"\nretenu : degré {degree_best}, λ = {lambda_best:.3g}, RMSE de validation "
      f"{validation_rmse[i_best, j_best]:.2f}")

phi_best = polynomial_basis(degree_best, X_train)
w_best = ridge_equation(phi_best(X_train), y_train, lambda_best)
print(f"R² de test du modèle retenu : {r2(y_test, predict(phi_best(X_test), w_best)):.3f}")
print(f"R² de test du meilleur degré 3 vu en 2.2 : {max(r2_test_path):.3f}")

Aucune case des lignes « degré 2 » et « degré 3 » ne descend sous 55,1 : le minimum de la grille est
sur la ligne « degré 1 ». Quel que soit $\lambda$, les colonnes ajoutées par les degrés supérieurs
coûtent plus en erreur de validation qu'elles ne rapportent, et ces 353 patients ne permettent donc
d'établir aucune courbure. Avec dix fois plus de patients, la réponse pourrait changer.

Le degré 3 pénalisé atteint pourtant 0,345 sur le test, contre 0,332 pour le modèle retenu. Ce
0,345 a été lu sur le test, il n'est donc pas comparable à un choix fait en validation, et l'écart
est minuscule là où celui de la validation, 55,07 contre 63,1, est franc.

## 2.4 Chaque analyse sanguine coûte une prise de sang

Les six analyses `s1` à `s6` ne sont pas gratuites : chacune se paie en réactifs et en temps de
laboratoire. La question posée au modèle change donc de nature : non plus « prédis au mieux »,
mais **« de combien de mesures as-tu réellement besoin ? »**

Le ridge ne sait pas répondre : il rétrécit les coefficients sans jamais les annuler. On remplace
la pénalité $\lVert w \rVert^2$ par la somme des valeurs absolues :

$$J(w) = \lVert Xw - y \rVert^2 + \alpha \sum_{j=1}^{n} |w_j| .$$

C'est le **lasso**. Cette pénalité n'est pas différentiable en zéro : il n'y a pas de forme close,
et on passe par un solveur, `Lasso` de scikit-learn.

**Q7.** Retirer des colonnes au modèle, c'est lui retirer de l'information. Le $R^2$ de test peut-il
*monter* quand on en retire ? Justifiez en une phrase.

**Votre réponse :**



In [ ]:
def lasso_cost(X, y, w, alpha):
    """||Xw - y||^2 + alpha * sum of |w_j|, the bias excluded from the penalty."""
    # TODO: the sum of squared errors, plus alpha times the sum of the absolute weights
    ...


# no closed form for this one: a solver finds the minimum, and it does beat the ridge solution
w_lasso = Lasso(alpha=0.2, max_iter=100000).fit(X_train, y_train)
w_lasso = np.r_[w_lasso.intercept_, w_lasso.coef_]
w_ridge_ref = ridge_equation(X_train, y_train, 0.2)
assert lasso_cost(X_train, y_train, w_lasso, 0.2) < lasso_cost(X_train, y_train, w_ridge_ref, 0.2)

def lasso_cost(X, y, w, alpha):
    """||Xw - y||^2 + alpha * sum of |w_j|, the bias excluded from the penalty."""
    return np.sum((design_matrix(X) @ w - y) ** 2) + alpha * np.sum(np.abs(w[1:]))


# no closed form for this one: a solver finds the minimum, and it does beat the ridge solution
w_lasso = Lasso(alpha=0.2, max_iter=100000).fit(X_train, y_train)
w_lasso = np.r_[w_lasso.intercept_, w_lasso.coef_]
w_ridge_ref = ridge_equation(X_train, y_train, 0.2)
assert lasso_cost(X_train, y_train, w_lasso, 0.2) < lasso_cost(X_train, y_train, w_ridge_ref, 0.2)

alphas = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0]

full = LinearRegression().fit(X_train, y_train)
rows = [{"α": 0.0, "mesures gardées": 10, "R² test": full.score(X_test, y_test),
         "lesquelles": ", ".join(feature_names)}]
for alpha in alphas:
    model = Lasso(alpha=alpha, max_iter=100000).fit(X_train, y_train)
    kept = [feature_names[j] for j in np.flatnonzero(model.coef_)]
    rows.append({"α": alpha, "mesures gardées": len(kept),
                 "R² test": model.score(X_test, y_test), "lesquelles": ", ".join(kept)})

assert rows[1]["mesures gardées"] == 9
assert rows[-1]["mesures gardées"] == 2
print(pd.DataFrame(rows).to_string(index=False, formatters={"R² test": "{:.3f}".format}))

**Q8.** La ligne $\alpha = 0$ garde les dix mesures `age, sex, bmi, bp, s1, s2, s3, s4, s5, s6` et
obtient 0,332 de $R^2$ de test ; la ligne $\alpha = 0{,}2$ garde `sex, bmi, bp, s2, s3, s5` et
obtient 0,343. Quelles analyses sanguines le laboratoire peut-il arrêter de commander ?

**Votre réponse :**



## 2.5 Rétrécir, ou annuler

Les deux pénalités se comportent différemment sur les deux mesures de cholestérol (`s1` le
total, `s2` le LDL), qui corrèlent à 0,889.

**Q9.** `s1` et `s2` corrèlent à 0,889 : leurs colonnes disent presque la même chose. Mettre un
poids de 2 sur `s1` seule, ou 1 sur chacune des deux : cela change-t-il la somme des valeurs
absolues ? Qu'en concluez-vous sur ce que le lasso va faire de cette paire ?

**Votre réponse :**



In [ ]:
def l2_penalty(w):
    # the penalty behind the ridge
    # TODO: the sum of the squared weights
    ...


def l1_penalty(w):
    # the penalty behind the lasso
    # TODO: the sum of the absolute weights
    ...


# each solution is the smaller one under its own penalty
assert l2_penalty(Ridge(alpha=0.2).fit(X_train, y_train).coef_) < l2_penalty(w_lasso[1:])
assert l1_penalty(w_lasso[1:]) < l1_penalty(Ridge(alpha=0.2).fit(X_train, y_train).coef_)

def l2_penalty(w):
    # the penalty behind the ridge
    return np.sum(w ** 2)


def l1_penalty(w):
    # the penalty behind the lasso
    return np.sum(np.abs(w))


# each solution is the smaller one under its own penalty
assert l2_penalty(Ridge(alpha=0.2).fit(X_train, y_train).coef_) < l2_penalty(w_lasso[1:])
assert l1_penalty(w_lasso[1:]) < l1_penalty(Ridge(alpha=0.2).fit(X_train, y_train).coef_)

ridge_path = np.array([Ridge(alpha=a).fit(X_train, y_train).coef_
                       for a in np.logspace(-2, 3, 40)])
# TODO: the same path for the lasso, over np.logspace(-3, 0.5, 40)
lasso_path = ...

assert lasso_path.shape == (40, 10)
assert (ridge_path == 0).sum() == 0

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for j, name in enumerate(feature_names):
    style = dict(lw=2.4) if name in ("s1", "s2") else dict(lw=1, alpha=0.55)
    axes[0].plot(np.logspace(-2, 3, 40), ridge_path[:, j], label=name, **style)
    axes[1].plot(np.logspace(-3, 0.5, 40), lasso_path[:, j], label=name, **style)
for ax, title in zip(axes, ["ridge : λ", "lasso : α"]):
    ax.set_xscale("log")
    ax.axhline(0, color="black", lw=1)
    ax.set_xlabel(title)
axes[0].set_ylabel("coefficient")
axes[1].legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

print(f"coefficients exactement nuls ; ridge : {(ridge_path == 0).sum()}, "
      f"lasso : {(lasso_path == 0).sum()} sur {lasso_path.size}")
print()
for alpha in (0.05, 0.1, 0.2, 0.3):
    coefficients = Lasso(alpha=alpha, max_iter=100000).fit(X_train, y_train).coef_
    print(f"lasso α = {alpha:<5} s1 (cholestérol total) {coefficients[4]:8.1f}, "
          f"s2 (LDL) {coefficients[5]:8.1f}")

Le lasso garde `s1` et annule `s2` à $\alpha = 0{,}05$ et $0{,}1$, puis fait exactement l'inverse à
$\alpha = 0{,}2$. Un zéro du lasso ne dit donc pas « cette mesure n'a pas d'effet » ; il dit « cette
colonne n'a pas été retenue, à ce réglage ». Entre ces lignes, seul $\alpha$ a changé.

Le compte des zéros résume l'opposition : aucun côté ridge, 130 sur 400 côté lasso. Au clinicien qui
veut comprendre, le ridge garde les deux mesures et répond qu'elles portent le même signal ; au
laboratoire qui veut réduire ses coûts, les zéros du lasso sont exactement la réponse cherchée.